In [1]:
import numpy as np
from pynq import Overlay, allocate

# 1. Load the Overlay
# Ensure your .bit and .hwh files have the exact same name and are in the same directory.
print("Loading overlay...")
overlay = Overlay("bitstream_test/design_1_wrapper.bit")

# Access the DMAs by their block design names (adjust these if you named them differently in Vivado)
dma_0 = overlay.axi_dma_0
dma_1 = overlay.axi_dma_1

# 2. Define Dimensions and Allocate Memory
N = 16

# Use pynq.allocate to ensure the memory is contiguous and accessible by the PL
# Using int16 as requested. 
in_buffer_A = allocate(shape=(N, N), dtype=np.int16)
in_buffer_B = allocate(shape=(N, N), dtype=np.int16)

# Note: Depending on your RTL, the output of a 16x16 int16 multiplication might require 
# a larger accumulator (like int32) to prevent overflow. Adjust dtype here if necessary.
out_buffer_C = allocate(shape=(N, N), dtype=np.int16) 

# 3. Initialize Matrices with Small Values
# Using small random values between 1 and 5 for easy manual checking if needed
np.copyto(in_buffer_A, np.random.randint(1, 5, size=(N, N), dtype=np.int16))
np.copyto(in_buffer_B, np.random.randint(1, 5, size=(N, N), dtype=np.int16))

# Compute the expected result on the PS (CPU) to act as our ground truth
expected_C = np.dot(in_buffer_A, in_buffer_B)

print("Starting DMA transfers...")

# 1. Prime the receive channel FIRST so it is ready to catch incoming data
dma_0.recvchannel.transfer(out_buffer_C)

# 2. Trigger the sends
dma_0.sendchannel.transfer(in_buffer_A)
dma_1.sendchannel.transfer(in_buffer_B)

# 3. Wait for completion
dma_0.sendchannel.wait()
dma_1.sendchannel.wait()
print("Sends completed...")

dma_0.recvchannel.wait() # Hopefully, it passes this now!
print("Transfers complete.\n")


# 6. Print and Compare Results
print("--- Received Data from Systolic Array (PL) ---")
print(out_buffer_C)

print("\n--- Expected Data from CPU (PS) ---")
print(expected_C)

# Verification
if np.array_equal(out_buffer_C, expected_C):
    print("\n✅ SUCCESS: Hardware and Software results match perfectly!")
else:
    print("\n❌ FAILURE: Data mismatch detected.")
    # Optional: Print the difference to help debug RTL alignment or overflow issues
    print("Difference (PL - PS):")
    print(out_buffer_C - expected_C)

# 7. Clean up Memory
in_buffer_A.freebuffer()
in_buffer_B.freebuffer()
out_buffer_C.freebuffer()

Loading overlay...


Starting DMA transfers...


RuntimeError: DMA channel not started

In [11]:
import time
import numpy as np
from pynq import Overlay, allocate

# ─────────────────────────────────────────────
# 1. Load Overlay
# ─────────────────────────────────────────────
print("Loading overlay...")
overlay = Overlay("bitstream_test/design_1_wrapper.bit")

dma_0 = overlay.axi_dma_0
dma_1 = overlay.axi_dma_1

# ─────────────────────────────────────────────
# 2. Reset & Start DMA (VERY IMPORTANT FIX)
# ─────────────────────────────────────────────
print("Resetting DMA...")

dma_0.sendchannel.stop()
dma_0.recvchannel.stop()
dma_1.sendchannel.stop()

time.sleep(0.01)

dma_0.sendchannel.start()
dma_0.recvchannel.start()
dma_1.sendchannel.start()

print("DMA started.\n")

# ─────────────────────────────────────────────
# 3. Allocate Buffers
# ─────────────────────────────────────────────
N = 16

in_buffer_A = allocate((N, N), dtype=np.int16)
in_buffer_B = allocate((N, N), dtype=np.int16)

# ⚠️ Use int32 to avoid overflow (recommended)
out_buffer_C = allocate((N, N), dtype=np.int32)

# ─────────────────────────────────────────────
# 4. Initialize Data
# ─────────────────────────────────────────────
np.copyto(in_buffer_A, np.random.randint(1, 5, (N, N), dtype=np.int16))
np.copyto(in_buffer_B, np.random.randint(1, 5, (N, N), dtype=np.int16))

expected_C = np.dot(in_buffer_A, in_buffer_B)

# ─────────────────────────────────────────────
# 5. Start Transfers
# ─────────────────────────────────────────────
print("Starting DMA transfers...")

# Start receive FIRST
dma_0.recvchannel.transfer(out_buffer_C)

# Debug before send
print("Before transfer:")
print("DMA0 running:", dma_0.sendchannel.running)
print("DMA1 running:", dma_1.sendchannel.running)

# Send inputs
dma_0.sendchannel.transfer(in_buffer_A)
dma_1.sendchannel.transfer(in_buffer_B)

# Debug after send
print("After transfer:")
print("DMA0 running:", dma_0.sendchannel.running)
print("DMA1 running:", dma_1.sendchannel.running)

# Small delay for handshake stability
time.sleep(0.01)

# ─────────────────────────────────────────────
# 6. Safe Wait for Send Completion
# ─────────────────────────────────────────────
try:
    dma_0.sendchannel.wait()
    dma_1.sendchannel.wait()
    print("Sends completed...")
except RuntimeError as e:
    print("❌ Send Error:", e)

# ─────────────────────────────────────────────
# 7. Poll Receive Channel (with timeout)
# ─────────────────────────────────────────────
timeout = 10
start = time.time()

while time.time() - start < timeout:
    s2mm_sr = dma_0.mmio.read(0x34)
    s2mm_len = dma_0.mmio.read(0x58)

    print(f"S2MM Status: {hex(s2mm_sr)} | Bytes received: {s2mm_len}")

    if dma_0.recvchannel.idle:
        print("✅ Receive Done!")
        break

    time.sleep(1)

else:
    print("❌ TIMEOUT!")

    print("\n--- DMA_0 Register Dump ---")
    for offset in [
        0x00, 0x04, 0x08, 0x18, 0x1C,
        0x30, 0x34, 0x38, 0x48, 0x58
    ]:
        print(f"Offset {hex(offset)}: {hex(dma_0.mmio.read(offset))}")

# Ensure receive completion
try:
    dma_0.recvchannel.wait()
except RuntimeError as e:
    print("❌ Receive Error:", e)

print("Transfers complete.\n")
    

    # ─────────────────────────────────────────────
# 8. Results
# ─────────────────────────────────────────────
print("--- PL Output ---")
print(out_buffer_C)

# 🔥 Transpose of PL (actual hardware intention)
pl_transpose = out_buffer_C.T

print("\n--- Transposed PL Output ---")
print(pl_transpose)

print("\n--- Expected (PS) ---")
print(expected_C)

# Compare ORIGINAL
if np.array_equal(out_buffer_C, expected_C):
    print("\n✅ SUCCESS: Direct match!")
    
# Compare TRANSPOSE
elif np.array_equal(pl_transpose, expected_C):
    print("\n✅ SUCCESS: Transpose match! (Your hardware outputs C^T)")
    
else:
    print("\n❌ FAILURE: No match.")
    print("\nDifference (PL^T - Expected):")
    print(pl_transpose - expected_C)

# ─────────────────────────────────────────────
# 9. Cleanup
# ─────────────────────────────────────────────
in_buffer_A.freebuffer()
in_buffer_B.freebuffer()
out_buffer_C.freebuffer()

Loading overlay...
Resetting DMA...
DMA started.

Starting DMA transfers...
Before transfer:
DMA0 running: True
DMA1 running: True
After transfer:
DMA0 running: True
DMA1 running: True
Sends completed...
S2MM Status: 0x1002 | Bytes received: 1024
✅ Receive Done!
Transfers complete.

--- PL Output ---
[[ 99  87 109 120 102 103 113  91 103  98 112  98  97  97 114 108]
 [ 81  78  92 108  94  95 106  83  86  95 101  84  93  82 104  97]
 [ 84  83 103 112 102  90 112  90  85  98  98  86 102  91  97 104]
 [102 101 117 132 115 118 126 101  99 116 123 107 108 101 112 110]
 [120 107 141 145 123 122 145 112 117 125 141 115 127 117 132 126]
 [ 92  91 113 121 104 104 112  89  91 106 117  86 111  91 104  97]
 [ 95  89 109 123 112 106 104  98  85 109 105 100  91  89  98 105]
 [ 99  96 120 117 106 108 119  90  81 112 121  91 113  99 108  98]
 [101  98 122 129 117 121 123 105  89 118 121 103 106 100 114 113]
 [ 95 103 124 137 121 124 124 104  97 122 127 107 105 101 115 116]
 [106 103 129 137 121 113 13

In [5]:
import time
import numpy as np
from pynq import Overlay, allocate

# ─────────────────────────────────────────────
# 1. Load Overlay
# ─────────────────────────────────────────────
print("Loading overlay...")
overlay = Overlay("bitstream_test/design_1_wrapper.bit")

dma_0 = overlay.axi_dma_0
dma_1 = overlay.axi_dma_1

# ─────────────────────────────────────────────
# 2. Reset & Start DMA
# ─────────────────────────────────────────────
print("Resetting DMA...")

dma_0.sendchannel.stop()
dma_0.recvchannel.stop()
dma_1.sendchannel.stop()

time.sleep(0.01)

dma_0.sendchannel.start()
dma_0.recvchannel.start()
dma_1.sendchannel.start()

print("DMA started.\n")

# ─────────────────────────────────────────────
# 3. Allocate Buffers
# ─────────────────────────────────────────────
N = 16

in_buffer_A = allocate((N, N), dtype=np.int16)
in_buffer_B = allocate((N, N), dtype=np.int16)
out_buffer_C = allocate((N, N), dtype=np.int32)

# ─────────────────────────────────────────────
# 4. Initialize Data (IDENTITY TEST)
# ─────────────────────────────────────────────
# Random A
np.copyto(in_buffer_A, np.random.randint(1, 10, (N, N), dtype=np.int16))

# Identity B
in_buffer_B[:] = np.eye(N, dtype=np.int16)

# Expected result
expected_C = in_buffer_A.copy()

print("Input A:")
print(in_buffer_A)

print("\nIdentity Matrix B:")
print(in_buffer_B)

# ─────────────────────────────────────────────
# 5. Start Transfers
# ─────────────────────────────────────────────
print("\nStarting DMA transfers...")

dma_0.recvchannel.transfer(out_buffer_C)

dma_0.sendchannel.transfer(in_buffer_A)
dma_1.sendchannel.transfer(in_buffer_B)

time.sleep(0.01)

dma_0.sendchannel.wait()
dma_1.sendchannel.wait()
print("Sends completed...")

# ─────────────────────────────────────────────
# 6. Wait for Receive
# ─────────────────────────────────────────────
dma_0.recvchannel.wait()
print("Transfers complete.\n")

# ─────────────────────────────────────────────
# 7. Results
# ─────────────────────────────────────────────
print("--- PL Output ---")
print(out_buffer_C)

print("\n--- Expected (A itself) ---")
print(expected_C)

# Verification
if np.array_equal(out_buffer_C, expected_C):
    print("\n✅ SUCCESS: Identity test passed!")
else:
    print("\n❌ FAILURE: Identity test failed.")
    print("Difference:")
    print(out_buffer_C - expected_C)

# ─────────────────────────────────────────────
# 8. Cleanup
# ─────────────────────────────────────────────
in_buffer_A.freebuffer()
in_buffer_B.freebuffer()
out_buffer_C.freebuffer()

Loading overlay...
Resetting DMA...
DMA started.

Input A:
[[6 2 5 4 3 5 1 2 5 5 2 4 9 6 1 4]
 [4 5 1 9 9 2 5 3 5 8 9 3 1 7 4 7]
 [2 8 3 7 8 9 9 5 2 2 7 3 7 9 4 8]
 [1 4 2 6 2 9 6 9 9 2 1 8 5 8 2 6]
 [4 1 8 6 6 1 2 9 8 6 6 2 1 4 8 4]
 [4 8 3 1 7 3 8 1 9 7 6 3 5 5 7 4]
 [8 8 5 1 9 4 3 6 4 6 6 3 5 4 6 8]
 [7 1 2 9 6 6 2 4 3 2 7 7 3 3 7 6]
 [3 2 7 6 7 7 7 7 7 5 8 1 8 8 3 7]
 [8 5 7 9 2 3 2 1 6 1 4 5 5 2 3 5]
 [4 2 3 1 6 9 7 2 9 8 2 8 1 4 4 6]
 [4 9 5 5 3 9 8 8 8 3 1 2 4 1 1 5]
 [6 5 4 2 3 3 4 7 9 8 1 6 5 1 9 6]
 [5 4 9 4 3 7 9 2 4 3 9 3 9 5 5 2]
 [4 8 8 6 4 5 1 5 8 4 3 2 4 6 5 9]
 [9 6 9 4 2 7 7 5 1 8 7 4 7 6 2 6]]

Identity Matrix B:
[[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 

In [7]:
import time
import numpy as np
from pynq import Overlay, allocate

# ─────────────────────────────────────────────
# 1. Load Overlay
# ─────────────────────────────────────────────
print("Loading overlay...")
overlay = Overlay("bitstream_test/design_1_wrapper.bit")

dma_0 = overlay.axi_dma_0
dma_1 = overlay.axi_dma_1

# ─────────────────────────────────────────────
# 2. Reset & Start DMA
# ─────────────────────────────────────────────
print("Resetting DMA...")

dma_0.sendchannel.stop()
dma_0.recvchannel.stop()
dma_1.sendchannel.stop()

time.sleep(0.01)

dma_0.sendchannel.start()
dma_0.recvchannel.start()
dma_1.sendchannel.start()

print("DMA started.\n")

# ─────────────────────────────────────────────
# 3. Allocate Buffers
# ─────────────────────────────────────────────
N = 16

in_buffer_A = allocate((N, N), dtype=np.int16)
in_buffer_B = allocate((N, N), dtype=np.int16)
out_buffer_C = allocate((N, N), dtype=np.int32)

# ─────────────────────────────────────────────
# 4. Initialize Data (IDENTITY × IDENTITY)
# ─────────────────────────────────────────────
in_buffer_A[:] = np.eye(N, dtype=np.int16)
in_buffer_B[:] = np.eye(N, dtype=np.int16)

expected_C = np.eye(N, dtype=np.int32)

print("Matrix A (Identity):")
print(in_buffer_A)

print("\nMatrix B (Identity):")
print(in_buffer_B)

# ─────────────────────────────────────────────
# 5. Start Transfers
# ─────────────────────────────────────────────
print("\nStarting DMA transfers...")

dma_0.recvchannel.transfer(out_buffer_C)

dma_0.sendchannel.transfer(in_buffer_A)
dma_1.sendchannel.transfer(in_buffer_B)

time.sleep(0.01)

dma_0.sendchannel.wait()
dma_1.sendchannel.wait()
print("Sends completed...")

# ─────────────────────────────────────────────
# 6. Wait for Receive
# ─────────────────────────────────────────────
dma_0.recvchannel.wait()
print("Transfers complete.\n")

# ─────────────────────────────────────────────
# 7. Results
# ─────────────────────────────────────────────
print("--- PL Output ---")
print(out_buffer_C)

print("\n--- Expected (Identity) ---")
print(expected_C)

# Verification
if np.array_equal(out_buffer_C, expected_C):
    print("\n✅ SUCCESS: Identity × Identity test passed!")
else:
    print("\n❌ FAILURE: Identity × Identity test failed.")
    print("Difference:")
    print(out_buffer_C - expected_C)

# ─────────────────────────────────────────────
# 8. Cleanup
# ─────────────────────────────────────────────
in_buffer_A.freebuffer()
in_buffer_B.freebuffer()
out_buffer_C.freebuffer()

Loading overlay...
Resetting DMA...
DMA started.

Matrix A (Identity):
[[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]]

Matrix B (Identity):
[[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 0 0

In [12]:
import time
import numpy as np
from pynq import Overlay, allocate

# ─────────────────────────────────────────────
# 1. Load Overlay
# ─────────────────────────────────────────────
print("Loading overlay...")
overlay = Overlay("bitstream_test/design_1_wrapper.bit")

dma_0 = overlay.axi_dma_0
dma_1 = overlay.axi_dma_1

# ─────────────────────────────────────────────
# 2. Reset & Start DMA
# ─────────────────────────────────────────────
print("Resetting DMA...")

dma_0.sendchannel.stop()
dma_0.recvchannel.stop()
dma_1.sendchannel.stop()

time.sleep(0.01)

dma_0.sendchannel.start()
dma_0.recvchannel.start()
dma_1.sendchannel.start()

print("DMA started.\n")

# ─────────────────────────────────────────────
# 3. Allocate Buffers
# ─────────────────────────────────────────────
N = 16

in_buffer_A = allocate((N, N), dtype=np.int16)
in_buffer_B = allocate((N, N), dtype=np.int16)
out_buffer_C = allocate((N, N), dtype=np.int32)

# ─────────────────────────────────────────────
# 4. Initialize Data
# ─────────────────────────────────────────────
# A = Identity
in_buffer_A[:] = np.eye(N, dtype=np.int16)

# B = Random
np.copyto(in_buffer_B, np.random.randint(1, 5, (N, N), dtype=np.int16))

# Expected result = B
expected_C = in_buffer_B.copy()

print("Matrix A (Identity):")
print(in_buffer_A)

print("\nMatrix B (Random):")
print(in_buffer_B)

# ─────────────────────────────────────────────
# 5. Start Transfers
# ─────────────────────────────────────────────
print("\nStarting DMA transfers...")

dma_0.recvchannel.transfer(out_buffer_C)

dma_0.sendchannel.transfer(in_buffer_A)
dma_1.sendchannel.transfer(in_buffer_B)

time.sleep(0.01)

dma_0.sendchannel.wait()
dma_1.sendchannel.wait()
print("Sends completed...")

# ─────────────────────────────────────────────
# 6. Wait for Receive
# ─────────────────────────────────────────────
dma_0.recvchannel.wait()
print("Transfers complete.\n")

# ─────────────────────────────────────────────
# 7. Results
# ─────────────────────────────────────────────
print("--- PL Output ---")
print(out_buffer_C)

pl_transpose = out_buffer_C.T

print("\n--- Transposed PL Output ---")
print(pl_transpose)

print("\n--- Expected (B itself) ---")
print(expected_C)

# Check direct
if np.array_equal(out_buffer_C, expected_C):
    print("\n✅ SUCCESS: Direct match!")

# Check transpose
elif np.array_equal(pl_transpose, expected_C):
    print("\n✅ SUCCESS: Transpose match! (Hardware outputs C^T)")

else:
    print("\n❌ FAILURE")
    print("\nDifference (PL^T - Expected):")
    print(pl_transpose - expected_C)

# ─────────────────────────────────────────────
# 8. Cleanup
# ─────────────────────────────────────────────
in_buffer_A.freebuffer()
in_buffer_B.freebuffer()
out_buffer_C.freebuffer()

Loading overlay...
Resetting DMA...
DMA started.

Matrix A (Identity):
[[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]]

Matrix B (Random):
[[3 2 2 1 3 2 3 4 3 1 2 2 2 2 2 1]
 [3 4 2 4 4 4 2 3 4 4 4 3 1 1 2 1]
 [1 4 3 3 2 3 1 3 1 2 4 1 3 4 3 2]
 [3 3 1 3 2 1 1 2 1 4 2 4 1 4 4 4]
 [3 4 1 3 4 2 3 1 1 3 1 2 3 3 4 2]
 [1 4 2 3 3 1 3 1 1 3 1 1 2 1 3 2]
 [1 2 2 3 1 2 4 3 1 4 4 1 4 2 3 4]
 [2 4 2 4 3 2 3 1 3 4 2 2 4 1 4 3]
 [3 4 2 3 4 3 3 1 1 4 4 4 1 2 2 4]
 [2 2 4 3 4 4 4 4 2 1 2 3 2 1 2 4

In [14]:
import time
import numpy as np
from pynq import Overlay, allocate

# ─────────────────────────────────────────────
# 1. Load Overlay
# ─────────────────────────────────────────────
print("Loading overlay...")
overlay = Overlay("bitstream_test/design_1_wrapper.bit")

dma_0 = overlay.axi_dma_0
dma_1 = overlay.axi_dma_1

# ─────────────────────────────────────────────
# 2. Reset & Start DMA
# ─────────────────────────────────────────────
print("Resetting DMA...")

dma_0.sendchannel.stop()
dma_0.recvchannel.stop()
dma_1.sendchannel.stop()

time.sleep(0.01)

dma_0.sendchannel.start()
dma_0.recvchannel.start()
dma_1.sendchannel.start()

print("DMA started.\n")

# ─────────────────────────────────────────────
# 3. Allocate Buffers
# ─────────────────────────────────────────────
N = 16

in_buffer_A = allocate((N, N), dtype=np.int16)
in_buffer_B = allocate((N, N), dtype=np.int16)
out_buffer_C = allocate((N, N), dtype=np.int32)

# ─────────────────────────────────────────────
# 4. Initialize Data (NON-IDENTITY TEST)
# ─────────────────────────────────────────────
np.copyto(in_buffer_A, np.random.randint(1, 5, (N, N), dtype=np.int16))
np.copyto(in_buffer_B, np.random.randint(1, 5, (N, N), dtype=np.int16))

# 🔥 Expected = B^T * A^T
expected_C = np.dot(in_buffer_B.T, in_buffer_A)

print("Matrix A:")
print(in_buffer_A)

print("\nMatrix B:")
print(in_buffer_B)

# ─────────────────────────────────────────────
# 5. Start Transfers
# ─────────────────────────────────────────────
print("\nStarting DMA transfers...")

dma_0.recvchannel.transfer(out_buffer_C)

dma_0.sendchannel.transfer(in_buffer_A)
dma_1.sendchannel.transfer(in_buffer_B)

time.sleep(0.01)

dma_0.sendchannel.wait()
dma_1.sendchannel.wait()
print("Sends completed...")

# ─────────────────────────────────────────────
# 6. Wait for Receive
# ─────────────────────────────────────────────
dma_0.recvchannel.wait()
print("Transfers complete.\n")

# ─────────────────────────────────────────────
# 7. Results
# ─────────────────────────────────────────────
print("--- PL Output ---")
print(out_buffer_C)

print("\n--- Expected (B^T × A^T) ---")
print(expected_C)

# Optional: also check transpose of PL
pl_transpose = out_buffer_C.T

print("\n--- Transposed PL Output ---")
print(pl_transpose)

# ─────────────────────────────────────────────
# 8. Verification
# ─────────────────────────────────────────────
if np.array_equal(out_buffer_C, expected_C):
    print("\n✅ SUCCESS: Matches B^T × A^T")

elif np.array_equal(pl_transpose, expected_C):
    print("\n✅ SUCCESS: Transpose matches B^T × A^T")

else:
    print("\n❌ FAILURE")
    print("\nDifference (PL - Expected):")
    print(out_buffer_C - expected_C)

# ─────────────────────────────────────────────
# 9. Cleanup
# ─────────────────────────────────────────────
in_buffer_A.freebuffer()
in_buffer_B.freebuffer()
out_buffer_C.freebuffer()

Loading overlay...
Resetting DMA...
DMA started.

Matrix A:
[[1 2 3 1 2 4 2 1 3 3 3 4 4 2 4 3]
 [2 1 3 4 3 3 1 2 2 4 3 3 4 4 3 1]
 [1 1 4 2 4 1 1 4 2 1 1 1 4 2 3 2]
 [1 1 2 3 4 3 1 3 1 2 1 3 1 4 3 4]
 [2 1 4 1 2 1 4 4 2 4 3 3 4 3 3 3]
 [3 3 2 1 4 3 4 3 3 4 1 2 4 3 3 1]
 [2 2 1 1 4 4 1 1 2 1 3 3 1 1 2 1]
 [1 2 3 2 1 4 3 1 3 4 4 1 3 3 1 4]
 [4 1 4 1 2 2 1 1 2 1 1 4 4 2 2 2]
 [4 3 3 2 4 1 1 1 2 2 3 3 3 2 1 3]
 [2 1 1 1 1 3 2 2 2 4 4 2 3 1 3 2]
 [1 2 4 1 4 1 2 3 1 2 1 1 4 2 4 4]
 [3 2 1 2 1 4 1 1 2 4 2 2 1 3 3 4]
 [1 2 2 4 3 1 3 4 3 4 3 4 2 3 2 4]
 [4 3 4 3 2 2 2 2 4 3 4 4 2 1 3 4]
 [2 2 2 1 1 1 3 2 4 1 4 2 3 3 4 1]]

Matrix B:
[[1 1 2 2 3 3 1 4 4 2 2 4 1 4 4 1]
 [1 3 2 4 4 3 1 2 3 1 2 1 3 4 4 1]
 [3 2 1 1 3 1 4 4 1 3 1 4 3 3 2 2]
 [3 2 1 1 1 4 4 4 3 2 3 1 4 4 4 3]
 [3 1 2 4 2 1 4 4 3 4 2 4 1 3 1 4]
 [4 1 3 4 1 2 1 1 4 1 4 2 4 4 3 4]
 [2 2 2 3 2 2 4 2 4 2 3 3 2 4 1 3]
 [4 4 4 2 3 1 1 1 1 2 2 2 4 2 2 1]
 [4 4 1 3 2 4 1 1 2 3 1 4 1 4 3 4]
 [2 1 2 4 2 1 4 3 4 4 3 2 4 1 1 2]
 [3 4 4 4 2 3 2 4 